# Step 3: Analysis

Analyzes the 28-day test against the design in `docs/experiment_design.md`: primary and secondary metrics, the two diagnostic ratios, guardrails, CUPED, a peeking demonstration, and effects by traffic source. Every estimate is also compared with the true effect from the simulation's ground truth.

Run `01_simulate_and_check_metrics` first so the experiment tables exist.

In [1]:
# Imports and paths.
import json
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import analysis as an

SQL = ROOT / "sql"
con = duckdb.connect(str(ROOT / "data" / "marketplace.duckdb"))

In [2]:
# Per-user metrics from SQL, split by arm. All analysis below uses the user as the unit.
con.execute((SQL / "01_user_metrics.sql").read_text())
um = con.execute("SELECT * FROM user_metrics").df()
num_cols = ["impressions", "clicks", "applications", "funded", "revenue", "prior_visits"]
um[num_cols] = um[num_cols].astype(float)
t, c = um[um["variant"] == 1], um[um["variant"] == 0]

srm = con.execute((SQL / "04_srm_counts.sql").read_text()).df()
print(f"Users: control {len(c):,}, treatment {len(t):,}; SRM p-value {stats.chisquare(srm['users']).pvalue:.3f}")

Users: control 99,926, treatment 100,074; SRM p-value 0.741


In [3]:
# Primary and secondary metrics, with 95% CIs, next to the true effect from ground truth.
# A well-behaved analysis should produce intervals that contain the true value.
truth = con.execute("""
SELECT AVG(true_funded_t) / AVG(true_funded_c) - 1   AS funded_true_lift,
       AVG(true_revenue_t) / AVG(true_revenue_c) - 1 AS revenue_true_lift
FROM ground_truth
""").df().iloc[0]

results = {
    "funded_per_user": an.mean_diff(t["funded"], c["funded"]),
    "revenue_per_lead": an.mean_diff(t["revenue"], c["revenue"]),
}
true_lift = {"funded_per_user": truth["funded_true_lift"], "revenue_per_lead": truth["revenue_true_lift"]}

def show(res):
    df = pd.DataFrame(res).T[["control", "treatment", "rel_lift", "rel_ci_low", "rel_ci_high", "p_value"]]
    return df.astype(float).round(4)

primary = show(results)
primary["true_lift"] = pd.Series(true_lift).round(4)
primary["ci_contains_truth"] = (primary["rel_ci_low"] <= primary["true_lift"]) & (primary["true_lift"] <= primary["rel_ci_high"])
print(primary.to_string())

                  control  treatment  rel_lift  rel_ci_low  rel_ci_high  p_value  true_lift  ci_contains_truth
funded_per_user    0.1566     0.1603    0.0233      0.0009       0.0463   0.0414     0.0345               True
revenue_per_lead  28.7394    29.2499    0.0178     -0.0065       0.0426   0.1520     0.0310               True


In [4]:
# Diagnostic ratio metrics with the delta method, which keeps the user as the unit of analysis.
# The naive standard error treats every impression or click as independent. It can understate
# uncertainty when a user's impressions are correlated; se_ratio shows how much here.
ratios = {
    "ctr": ("clicks", "impressions"),
    "click_to_funded": ("funded", "clicks"),
}
rows = {}
for name, (num, den) in ratios.items():
    r = an.ratio_diff(t[num], t[den], c[num], c[den])
    r["naive_se"] = an.naive_ratio_se(t[num], t[den], c[num], c[den])
    rows[name] = r
diag = pd.DataFrame(rows).T[["control", "treatment", "rel_lift", "rel_ci_low", "rel_ci_high", "p_value", "se", "naive_se"]]
diag["se_ratio"] = diag["se"] / diag["naive_se"]
print(diag.astype(float).round(4).to_string())

                 control  treatment  rel_lift  rel_ci_low  rel_ci_high  p_value      se  naive_se  se_ratio
ctr               0.1514     0.1400   -0.0750     -0.0840      -0.0659      0.0  0.0007    0.0007    1.0327
click_to_funded   0.2069     0.2289    0.1063      0.0846       0.1284      0.0  0.0022    0.0022    1.0168


In [5]:
# Guardrails, judged against the thresholds fixed in the design document.
# Approval rate: the CI lower bound must stay above -1 percentage point.
appr = an.ratio_diff(t["funded"], t["applications"], c["funded"], c["applications"])
appr_pass = appr["ci_low"] > -0.01

# Partner lead share: no partner moves more than 2 points; the top two partners
# (by control share) together rise by no more than 3 points.
ls = con.execute((SQL / "03_partner_lead_share.sql").read_text()).df()
ls["change"] = ls["treatment_share"] - ls["control_share"]
top2 = ls.nlargest(2, "control_share")
max_move = ls["change"].abs().max()
top2_change = top2["change"].sum()

guardrails = pd.DataFrame({
    "value": [appr["diff"], max_move, top2_change],
    "threshold": ["CI low > -0.010", "<= 0.020", "<= 0.030"],
    "passes": [appr_pass, max_move <= 0.02, top2_change <= 0.03],
}, index=["approval_rate_change", "max_partner_share_move", "top2_share_change"])
print(f"Approval rate: {appr['control']:.4f} -> {appr['treatment']:.4f}, "
      f"change CI [{appr['ci_low']:+.4f}, {appr['ci_high']:+.4f}]")
print(f"Top two partners by control share: {', '.join(top2['partner_id'])}")
print(guardrails.to_string())

Approval rate: 0.3766 -> 0.4178, change CI [+0.0342, +0.0481]
Top two partners by control share: P05, P04
                           value        threshold  passes
approval_rate_change    0.041156  CI low > -0.010    True
max_partner_share_move  0.008456         <= 0.020    True
top2_share_change       0.016292         <= 0.030    True


In [6]:
# CUPED using pre-period visits. theta is estimated on both arms pooled; the adjusted
# outcome keeps the same treatment effect but has lower variance.
cuped_rows = {}
for metric in ["funded", "revenue"]:
    adj, theta = an.cuped_adjust(um[metric], um["prior_visits"])
    um[f"{metric}_cuped"] = adj
    raw = an.mean_diff(t[metric], c[metric])
    cup = an.mean_diff(um.loc[um["variant"] == 1, f"{metric}_cuped"], um.loc[um["variant"] == 0, f"{metric}_cuped"])
    cuped_rows[metric] = {
        "theta": theta,
        "raw_rel_lift": raw["rel_lift"], "raw_se": raw["se"], "raw_p": raw["p_value"],
        "cuped_rel_lift": cup["rel_lift"], "cuped_se": cup["se"], "cuped_p": cup["p_value"],
        "variance_reduction": 1 - (cup["se"] / raw["se"]) ** 2,
    }
print(pd.DataFrame(cuped_rows).T.astype(float).round(4).to_string())

          theta  raw_rel_lift  raw_se   raw_p  cuped_rel_lift  cuped_se  cuped_p  variance_reduction
funded   0.0229        0.0233  0.0018  0.0414          0.0230    0.0018   0.0441              0.0072
revenue  4.2335        0.0178  0.3564  0.1520          0.0174    0.3553   0.1598              0.0062


In [7]:
# Peeking: run 500 A/A tests on control users only (no true effect), checking significance
# after every day. With one look at the end, about 5% should be significant by chance.
# Stopping at the first significant day inflates that rate.
fpr_end, fpr_any = an.aa_peeking(c["funded"].to_numpy(), c["day"].to_numpy(), n_days=28, reps=500)
print(f"False positive rate, one look at day 28: {fpr_end:.1%}")
print(f"False positive rate, checking every day: {fpr_any:.1%}")

False positive rate, one look at day 28: 4.6%
False positive rate, checking every day: 29.8%


In [8]:
# Effect on the primary metric by traffic source. Segment results are exploratory: three
# comparisons raise the chance of a false positive, and the test was not powered per segment.
seg = {}
for source, grp in um.groupby("source"):
    r = an.mean_diff(grp.loc[grp["variant"] == 1, "funded"], grp.loc[grp["variant"] == 0, "funded"])
    r["users"] = len(grp)
    seg[source] = r
seg_df = pd.DataFrame(seg).T[["users", "control", "treatment", "rel_lift", "rel_ci_low", "rel_ci_high", "p_value"]]
print(seg_df.astype(float).round(4).to_string())

                    users  control  treatment  rel_lift  rel_ci_low  rel_ci_high  p_value
marketplace      109983.0   0.1505     0.1493   -0.0076     -0.0377       0.0234   0.6266
partner_channel   59973.0   0.1584     0.1628    0.0280     -0.0126       0.0703   0.1785
security_app      30044.0   0.1756     0.1952    0.1116      0.0548       0.1716   0.0001


In [9]:
# Decision, applying the rule from the design document, and results saved for the readout.
ship = (
    results["funded_per_user"]["p_value"] < 0.05 and results["funded_per_user"]["diff"] > 0
    and not (results["revenue_per_lead"]["p_value"] < 0.05 and results["revenue_per_lead"]["diff"] < 0)
    and bool(guardrails["passes"].all())
)
print("Decision:", "SHIP" if ship else "DO NOT SHIP")

out = {
    "primary": primary.reset_index().to_dict("records"),
    "diagnostics": diag.astype(float).reset_index().to_dict("records"),
    "guardrails": guardrails.reset_index().astype(str).to_dict("records"),
    "cuped": pd.DataFrame(cuped_rows).T.astype(float).reset_index().to_dict("records"),
    "peeking": {"fpr_end": fpr_end, "fpr_any": fpr_any},
    "segments": seg_df.astype(float).reset_index().to_dict("records"),
    "decision": "ship" if ship else "do not ship",
}
(ROOT / "docs" / "results.json").write_text(json.dumps(out, indent=2, default=str))

Decision: SHIP


3602

In [10]:
# Close the database connection. DuckDB allows only one process to hold the file open,
# so leaving it open blocks the next notebook from connecting.
con.close()